In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error,root_mean_squared_error,r2_score

dataset = fetch_california_housing(as_frame=True)


In [ ]:
x , y = dataset.data.copy() , dataset.target.copy()
y.head()

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
scaled = StandardScaler()
x_train_scaled = scaled.fit_transform(x_train)
x_test_scaled = scaled.transform(x_test)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lr_model = LinearRegression()
lr_model.fit(x_train_scaled, y_train)

y_pred = lr_model.predict(x_test_scaled)
print(y_pred)

MSE = mean_squared_error(y_test, y_pred)
R2S = r2_score(y_test,y_pred)
print("MSE:", MSE)
print("R2S:", R2S)


In [ ]:
from sklearn.linear_model import RidgeCV,Ridge

alphas = np.logspace(-3,4,100)

ridge_cv = RidgeCV(alphas=alphas,cv=10,scoring='neg_root_mean_squared_error')
ridge_cv.fit(x_train_scaled,y_train)

ridge_model = Ridge(alpha=ridge_cv.alpha_)
ridge_model.fit(x_train_scaled,y_train)

y_pred = ridge_model.predict(x_test_scaled)
print(y_pred)

MSE = mean_squared_error(y_test, y_pred)
R2S = r2_score(y_test,y_pred)
print("MSE:", MSE)
print("R2S:", R2S)

In [ ]:
from sklearn.linear_model import Lasso,LassoCV

lasso_cv = LassoCV(alphas=alphas,cv=10,max_iter=100000,n_jobs=-1)
lasso_cv.fit(x_train_scaled,y_train)

lasso_model = Lasso(alpha=lasso_cv.alpha_)
lasso_model.fit(x_train_scaled,y_train)

y_pred = lr_model.predict(x_test_scaled)
print(y_pred)

MSE = mean_squared_error(y_test, y_pred)
R2S = r2_score(y_test,y_pred)
print("MSE:", MSE)
print("R2S:", R2S)

In [ ]:
from sklearn.linear_model import ElasticNet,ElasticNetCV

l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]
en_cv = ElasticNetCV(l1_ratio=l1_ratios,alphas=np.logspace(-4, 1, 50),
    cv=10,
    max_iter=10000,
    n_jobs=-1)

en_cv.fit(x_train_scaled, y_train)

en_model = ElasticNet(alpha=en_cv.alpha_,l1_ratio=en_cv.l1_ratio_, max_iter=200000)
en_model.fit(x_train_scaled,y_train)

y_pred = lr_model.predict(x_test_scaled)
print(y_pred)

MSE = mean_squared_error(y_test, y_pred)
R2S = r2_score(y_test,y_pred)
print("MSE:", MSE)
print("R2S:", R2S)


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score,GridSearchCV

depth = range(2,50)

cv_train_scores=[]
for d in depth:
    dt = DecisionTreeRegressor(max_depth=d,random_state=42)
    dt.fit(x_train_scaled,y_train)
    train_score = cross_val_score(dt,x_train_scaled,y_train,scoring='neg_root_mean_squared_error',cv=5)
    cv_train_scores.append(-train_score.mean())

optimal_depth = depth[np.argmin(cv_train_scores)]

param_grid = {
    'min_samples_leaf':[2,3,5,7,8,9,10,20],
    'min_samples_split':[2],
    'max_depth':[optimal_depth-1,optimal_depth-2,optimal_depth,optimal_depth+1,optimal_depth+2]
}

grid_search_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=10,
    n_jobs=-1,
    scoring='neg_root_mean_squared_error',
)

grid_search_dt.fit(x_train_scaled,y_train)

dt_model = DecisionTreeRegressor(**grid_search_dt.best_params_,random_state=42)
dt_model.fit(x_train_scaled,y_train)

y_pred = dt_model.predict(x_test_scaled)
print(y_pred)

MSE = mean_squared_error(y_test, y_pred)
R2S = r2_score(y_test,y_pred)
print("MSE:", MSE)
print("R2S:", R2S)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

max_features_options = [0.1, 'sqrt', 0.33, 0.5, 0.7, 1.0]
n_trees_range = [10, 25, 50, 100, 150, 200, 300, 500]
oob_scores = []

for n in n_trees_range:
    rf = RandomForestRegressor(n_estimators=n,oob_score=True,n_jobs=-1,random_state=42)
    rf.fit(x_train_scaled,y_train)
    oob_scores.append(rf.oob_score_)

mf_oob_scores = []
for mf in max_features_options:
    rf = RandomForestRegressor(n_estimators=200,max_features=mf,oob_score=True,n_jobs=-1,random_state=42)
    rf.fit(x_train_scaled,y_train)
    mf_oob_scores.append(rf.oob_score_)

best_mf = max_features_options[np.argmax(mf_oob_scores)]

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_features=best_mf,
    n_jobs=-1,
    oob_score=True,
    min_samples_leaf=1,
    min_samples_split=2,
)

rf_model.fit(x_train_scaled,y_train)

y_pred = rf_model.predict(x_test_scaled)
print(y_pred)
pd.DataFrame(y_pred).head()
MSE = mean_squared_error(y_test, y_pred)
R2S = r2_score(y_test,y_pred)
print("MSE:", MSE)
print("R2S:", R2S)




In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Step 1: Find optimal n_estimators using early stopping logic
gbm_large = GradientBoostingRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42
)
gbm_large.fit(x_train_scaled, y_train)

test_deviance = []
for y_pred in gbm_large.staged_predict(x_test_scaled):
    test_deviance.append(mean_squared_error(y_test, y_pred))

best_n_trees = np.argmin(test_deviance) + 1

# Step 2: Fit final model with optimal n_estimators
gbm_model = GradientBoostingRegressor(
    n_estimators=best_n_trees,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    min_samples_leaf=10,
    random_state=42
)
gbm_model.fit(x_train_scaled, y_train)

# Step 3: Evaluate
y_pred_gb = gbm_model.predict(x_test_scaled)
MSE_GB = mean_squared_error(y_test, y_pred_gb)
R2S_GB = r2_score(y_test, y_pred_gb)

print("MSE:", MSE_GB)
print("R2S:", R2S_GB)

In [ ]:
pip install xgboost

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=18,
    min_child_weight=10,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    eval_metric='rmse',
    subsample=0.8
)

xgb_model.fit(x_train_scaled,y_train,
              eval_set=[(x_test_scaled,y_test)], verbose=False)

best_n_trees = xgb_model.best_iteration+1
y_pred_xgb = xgb_model.predict(x_test_scaled)
MSE_XGB = mean_squared_error(y_test, y_pred_xgb)
R2S_XGB = r2_score(y_test, y_pred_xgb)

print("XGBoost MSE:", MSE_XGB)
print("XGBoost R2S:", R2S_XGB)



In [ ]:
pip install lightgbm

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score

# Define the model with specified parameters
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,            # Number of boosting rounds
    learning_rate=0.05,           # Step size shrinkage
    max_depth=-1,                 # Maximum depth of tree (-1 means no limit, controlled by num_leaves)
    num_leaves=31,                # Maximum number of leaves in one tree
    min_child_samples=20,         # Minimum number of data needed in a child (leaf)
    min_child_weight=1e-3,        # Minimum sum hessian needed in a child (leaf)
    subsample=1.0,                # Subsample ratio of the training instances
    subsample_freq=0,             # Frequency of subsample, 0 means disable
    colsample_bytree=1.0,         # Subsample ratio of columns when constructing each tree
    reg_alpha=0.0,                # L1 regularization term on weights
    reg_lambda=0.0,               # L2 regularization term on weights
    random_state=42,              # Random number seed
    n_jobs=-1,                    # Number of parallel threads
    objective='regression',       # Loss function for regression
    importance_type='split',      # Feature importance type ('split' or 'gain')
    min_split_gain=0.0,           # Minimum loss reduction required to make a further partition
    class_weight=None,            # Weights associated with classes (ignored in regression)
    boosting_type='gbdt'          # Type of boosting algorithm ('gbdt', 'rf', 'dart', 'goss')
)

# Fit the model with early stopping using the test set as validation
lgb_model.fit(
    x_train_scaled, y_train,
    eval_set=[(x_test_scaled, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=0)]
)

# Predict
y_pred_lgb = lgb_model.predict(x_test_scaled)

# Evaluate
MSE_LGB = mean_squared_error(y_test, y_pred_lgb)
R2S_LGB = r2_score(y_test, y_pred_lgb)

print("LightGBM MSE:", MSE_LGB)
print("LightGBM R2S:", R2S_LGB)

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

# Define the model with specified parameters
svm_model = SVR(
    C=1.0,                # Regularization parameter
    epsilon=0.1,          # Epsilon in the epsilon-SVR model
    kernel='rbf',         # Kernel type ('linear', 'poly', 'rbf', 'sigmoid')
    gamma='scale',        # Kernel coefficient for 'rbf', 'poly', 'sigmoid'
    degree=3,             # Degree of the polynomial kernel function ('poly')
    coef0=0.0,            # Independent term in kernel function ('poly', 'sigmoid')
    shrinking=True,       # Whether to use the shrinking heuristic
    cache_size=200,       # Size of the kernel cache (in MB)
    max_iter=-1           # Maximum number of iterations (-1 means no limit)
)

# Fit the model
# Note: SVMs do not support early_stopping or eval_set natively in fit()
svm_model.fit(x_train_scaled, y_train)

# Predict
y_pred_svm = svm_model.predict(x_test_scaled)

# Evaluate
MSE_SVM = mean_squared_error(y_test, y_pred_svm)
R2S_SVM = r2_score(y_test, y_pred_svm)

print("SVM Regressor MSE:", MSE_SVM)
print("SVM Regressor R2S:", R2S_SVM)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Load Dataset
data = load_breast_cancer()
X = data.data
y = data.target

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Scale Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 1. Logistic Regression
# ============================================================
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    C=1.0,            # Inverse of regularization strength; smaller values specify stronger regularization.
    penalty='l2',     # Norm used in the penalization ('l1', 'l2', 'elasticnet', None).
    solver='lbfgs',   # Algorithm to use in the optimization problem ('newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga').
    max_iter=1000,    # Maximum number of iterations taken for the solvers to converge.
    random_state=42,  # Used when solver == 'sag', 'saga' or 'liblinear' to shuffle the data.
    n_jobs=-1         # Number of CPU cores used when parallelizing over classes if multi_class='ovr'.
)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")

# ============================================================
# 2. Support Vector Machine (SVM)
# ============================================================
from sklearn.svm import SVC

svm_model = SVC(
    C=1.0,            # Regularization parameter. The strength of the regularization is inversely proportional to C.
    kernel='rbf',     # Specifies the kernel type to be used in the algorithm ('linear', 'poly', 'rbf', 'sigmoid').
    gamma='scale',    # Kernel coefficient for 'rbf', 'poly' and 'sigmoid'. 'scale' uses 1 / (n_features * X.var()).
    degree=3,         # Degree of the polynomial kernel function ('poly'). Ignored by all other kernels.
    coef0=0.0,        # Independent term in kernel function. It is only significant in 'poly' and 'sigmoid'.
    shrinking=True,   # Whether to use the shrinking heuristic.
    cache_size=200,   # Specify the size of the kernel cache (in MB).
    probability=False,# Whether to enable probability estimates. Must be called before fit, slows down training.
    random_state=42   # Controls the pseudo random number generation for shuffling the data for probability estimates.
)
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")

# ============================================================
# 3. K-Nearest Neighbors (KNN)
# ============================================================
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(
    n_neighbors=5,    # Number of neighbors to use by default for kneighbors queries.
    weights='uniform',# Weight function used in prediction ('uniform', 'distance').
    algorithm='auto', # Algorithm used to compute the nearest neighbors ('ball_tree', 'kd_tree', 'brute', 'auto').
    leaf_size=30,     # Leaf size passed to BallTree or KDTree. This can affect the speed of the construction and query.
    p=2,              # Power parameter for the Minkowski metric. p=1 is Manhattan, p=2 is Euclidean.
    n_jobs=-1         # The number of parallel jobs to run for neighbors search.
)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)
print(f"KNN Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")

# ============================================================
# 4. Decision Tree Classification
# ============================================================
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    criterion='gini', # Function to measure the quality of a split ('gini', 'entropy', 'log_loss').
    splitter='best',  # Strategy used to choose the split at each node ('best', 'random').
    max_depth=None,   # The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure.
    min_samples_split=2, # The minimum number of samples required to split an internal node.
    min_samples_leaf=1,  # The minimum number of samples required to be at a leaf node.
    max_features=None,   # The number of features to consider when looking for the best split.
    random_state=42   # Controls the randomness of the estimator.
)
dt_model.fit(X_train_scaled, y_train)
y_pred_dt = dt_model.predict(X_test_scaled)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")

# ============================================================
# 5. Random Forest Classification
# ============================================================
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100, # The number of trees in the forest.
    criterion='gini', # Function to measure the quality of a split ('gini', 'entropy', 'log_loss').
    max_depth=None,   # The maximum depth of the tree.
    min_samples_split=2, # Minimum number of samples required to split an internal node.
    min_samples_leaf=1,  # Minimum number of samples required to be at a leaf node.
    max_features='sqrt', # Number of features to consider when looking for the best split.
    bootstrap=True,   # Whether bootstrap samples are used when building trees.
    oob_score=False,  # Whether to use out-of-bag samples to estimate the generalization score.
    n_jobs=-1,        # Number of jobs to run in parallel.
    random_state=42   # Controls both the randomness of the bootstrapping and the sampling of features.
)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")

# ============================================================
# 6. Gradient Boosting Classifier
# ============================================================
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=100, # The number of boosting stages to perform.
    learning_rate=0.1,# Learning rate shrinks the contribution of each tree by learning_rate.
    max_depth=3,      # Maximum depth of the individual regression estimators.
    min_samples_split=2, # Minimum number of samples required to split an internal node.
    min_samples_leaf=1,  # Minimum number of samples required to be at a leaf node.
    subsample=1.0,    # The fraction of samples to be used for fitting the individual base learners.
    max_features=None,   # The number of features to consider when looking for the best split.
    random_state=42   # Controls the random seed given to each Tree estimator.
)
gb_model.fit(X_train_scaled, y_train)
y_pred_gb = gb_model.predict(X_test_scaled)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}")

# ============================================================
# 7. XGBoost Classifier
# ============================================================
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=100,       # Number of gradient boosted trees.
    learning_rate=0.1,      # Boosting learning rate (eta).
    max_depth=6,            # Maximum tree depth for base learners.
    min_child_weight=1,     # Minimum sum of instance weight (hessian) needed in a child.
    gamma=0,                # Minimum loss reduction required to make a further partition on a leaf node.
    subsample=1.0,          # Subsample ratio of the training instances.
    colsample_bytree=1.0,   # Subsample ratio of columns when constructing each tree.
    reg_alpha=0.0,          # L1 regularization term on weights (alpha).
    reg_lambda=1.0,         # L2 regularization term on weights (lambda).
    use_label_encoder=False,# Deprecated in newer versions, but suppresses warning if set to False.
    eval_metric='logloss',  # Evaluation metric for validation data.
    random_state=42,        # Random number seed.
    n_jobs=-1               # Number of parallel threads used to run xgboost.
)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)
print(f"XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")

# ============================================================
# 8. LightGBM Classifier
# ============================================================
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=100,       # Number of boosted trees to fit.
    learning_rate=0.1,      # Boosting learning rate.
    max_depth=-1,           # Maximum tree depth. -1 means no limit.
    num_leaves=31,          # Maximum tree leaves for base learners.
    min_child_samples=20,   # Minimum number of data needed in a child (leaf).
    subsample=1.0,          # Subsample ratio of the training instances.
    colsample_bytree=1.0,   # Subsample ratio of columns when constructing each tree.
    reg_alpha=0.0,          # L1 regularization term on weights.
    reg_lambda=0.0,         # L2 regularization term on weights.
    random_state=42,        # Random number seed.
    n_jobs=-1,              # Number of parallel threads.
    verbose=-1,             # Controls the verbosity of the model.
    importance_type='split' # Type of feature importance ('split' or 'gain').
)
lgb_model.fit(X_train_scaled, y_train)
y_pred_lgb = lgb_model.predict(X_test_scaled)
print(f"LightGBM Accuracy: {accuracy_score(y_test, y_pred_lgb):.4f}")

# ============================================================
# 9. Naive Bayes (Gaussian)
# ============================================================
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB(
    var_smoothing=1e-9    # Portion of the largest variance of all features that is added to variances for calculation stability.
)
nb_model.fit(X_train_scaled, y_train)
y_pred_nb = nb_model.predict(X_test_scaled)
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")

# ============================================================
# 10. Linear Discriminant Analysis (LDA)
# ============================================================
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda_model = LinearDiscriminantAnalysis(
    solver='svd',         # Solver to use ('svd', 'lsqr', 'eigen').
    shrinkage=None,       # Shrinkage parameter ('auto' or float between 0 and 1).
    priors=None,          # Class priors. By default, the class proportions are inferred from the training data.
    n_components=None,    # Number of components (< n_classes - 1) for dimensionality reduction.
    store_covariance=False,# If True, explicitly compute the weighted within-class covariance matrix.
    tol=1e-4              # Threshold used for rank estimation in SVD solver.
)
lda_model.fit(X_train_scaled, y_train)
y_pred_lda = lda_model.predict(X_test_scaled)
print(f"LDA Accuracy: {accuracy_score(y_test, y_pred_lda):.4f}")